## Preprocessing

In diesem Notebook wird der Bilddatensatz für die späteren Modelle vorbereitet.
Dazu gehören:
- Label-Extraktion aus Dateinamen
- Auswahl/Begrenzung des Datensatzes (Top-5 Klassen, Limit pro Klasse)
- Vorverarbeitung und Speichern der Bilder in einem einheitlichen Format
- Train/Val/Test Split
- Feature-Extraktion für klassische ML-Modelle (DT/kNN/FFNN)
- Aufbau von `tf.data` Pipelines für ein CNN

<hr style="border: 1px solid #ccc; margin: 20px 0;" />

In [ ]:
import os
import random
import re
import random
import pathlib
from typing import Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm import tqdm
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA


### Globale Parameter

Hier definieren wir zentrale Parameter, die im gesamten Preprocessing verwendet werden:
- Bildgrößen je Modelltyp (CNN vs. klassische Modelle)
- Farbmodus
- Sampling (Top-5-Klassen, Limit pro Klasse)
- Split-Größen für Train/Val/Test

<hr style="border: 1px solid #ccc; margin: 20px 0;" />

In [ ]:
# Globale Parameter 
IMG_SIZE = 224           # CNN
IMG_SIZE_FLAT = 32       # Decision Tree / k-NN 
IMG_SIZE_FFNN = 64       # FFNN 
COLOR_MODE = "RGB"
LIMIT_PER_CLASS = 10000
PROCESS_TOP_5_ONLY = True

TEST_SIZE = 0.15         # Test-Anteil 
VAL_SIZE = 0.15          # Validation-Anteil 
RANDOM_STATE = 42
BATCH_SIZE = 32


### Pfade / Ordnerstruktur

Wir legen fest:
- wo die Rohdaten liegen
- wohin verarbeitete Bilder gespeichert werden
- wohin Splits und Feature-Dateien exportiert werden

<hr style="border: 1px solid #ccc; margin: 20px 0;" />


In [ ]:
import pathlib

# Verzeichnisse
base_path = pathlib.Path("./data/Plankton-Bilder")
processed_dir = pathlib.Path("../data/processed")
splits_dir = pathlib.Path("../data/splits")
features_dir = pathlib.Path("../data/features")

processed_dir.mkdir(parents=True, exist_ok=True)
splits_dir.mkdir(parents=True, exist_ok=True)
features_dir.mkdir(parents=True, exist_ok=True)

assert base_path.exists(), f"base_path existiert nicht: {base_path}"


### Label-Extraktion aus Dateinamen

Die Labels stecken im Dateinamen (z. B. `fs446_copepods_...png`).
Wir nutzen dafür Regex, damit die Label-Extraktion reproduzierbar und robust ist.

<hr style="border: 1px solid #ccc; margin: 20px 0;" />


In [ ]:
_LABEL_RE = re.compile(
    r"^[^_]*_(.*?)_roi",  # alles zwischen erstem '_' und '_roi'
    re.IGNORECASE
)
#Extrahiert das Label aus dem Dateinamen. Fallback: unknown_label.
def extract_label_from_filename(filename: str) -> str:
    name = pathlib.Path(filename).name
    m = _LABEL_RE.match(name)
    if m:
        return m.group(1).lower()
    return "unknown_label"


In [ ]:
# Tests für die Label-Extraktion
test_filenames = [
    "fs446_copepods_roi.3280386401.tif.png",
    "fs446_diatoms_test_roi0.2413470000.tif.png",
    "fs446_snow_roi3293627700.tif.png",
    "fs446_unknown_roi3337066001.tif.png",
    "fs446_pluteus_roi2421461900.tif.png",
    "weirdname.png",
]

print("🔍 Test Label-Extraktion:")
for fn in test_filenames:
    print(fn, "->", extract_label_from_filename(fn))

assert extract_label_from_filename("weirdname.png") == "unknown_label"
print("Label-Tests ok.")

### Metadaten sammeln + Dataset begrenzen

Wir lesen alle Bildpfade ein und bauen `df_raw`:
- Dateiname
- Pfad
- Label

Optional:
- nur Top-5 Klassen (`PROCESS_TOP_5_ONLY`)
- Limit pro Klasse (`LIMIT_PER_CLASS`)

<hr style="border: 1px solid #ccc; margin: 20px 0;" />


In [ ]:
def preprocess_images(image_paths) -> Tuple[pd.DataFrame, pd.DataFrame]:
    data = []
    for p in tqdm(image_paths, desc="Metadaten sammeln"):
        data.append({
            "filename": p.name,
            "path_raw": str(p.resolve()),
            "label": extract_label_from_filename(p.name),
        })

    df_raw = pd.DataFrame(data)
    if df_raw.empty:
        raise ValueError("Keine Bilder gefunden / df_raw ist leer.")

    # harte Validierung: unknown_label darf nicht im echten Dataset sein (wenn es zugelassen werden soll: diese Assertion entfernen und stattdessen filtern)
    if (df_raw["label"] == "unknown_label").any():
        bad = df_raw[df_raw["label"] == "unknown_label"].head(10)["filename"].tolist()
        raise ValueError(f"Label-Extraktion fehlgeschlagen für Dateien (Beispiele): {bad}")

    # Top-5
    if PROCESS_TOP_5_ONLY:
        top5 = df_raw["label"].value_counts().head(5).index
        df = df_raw[df_raw["label"].isin(top5)].copy()
    else:
        df = df_raw.copy()

    # Limit pro Klasse (reproduzierbar)
    if LIMIT_PER_CLASS is not None:
        df = (
            df.groupby("label", group_keys=False)
              .apply(lambda x: x.sample(min(len(x), LIMIT_PER_CLASS), random_state=RANDOM_STATE))
              .reset_index(drop=True)
        )

    return df_raw, df


### Validierung: Preprocessing-Output

Wir prüfen:
- keine `unknown_label`
- Shapes und Labelanzahl
- Verteilung pro Label (zur Einschätzung von Imbalance)

<hr style="border: 1px solid #ccc; margin: 20px 0;" />


In [ ]:
def validate_preprocessing(df_raw, df_limited):
    
    print("\nValidierung der Preprocessing-Ergebnisse:")

    # Überprüfung auf "unknown_label"
    assert "unknown_label" not in df_raw["label"].values, "Fehler: 'unknown_label' im Dataset gefunden!"
    assert "unknown_label" not in df_limited["label"].values, "Fehler: 'unknown_label' im gefilterten Dataset gefunden!"
    print("Keine 'unknown_label' im Dataset gefunden.")

    # Shape überprüfen
    print(f"Rohdaten: {df_raw.shape[0]} Bilder in {df_raw['label'].nunique()} Labels.")
    print(f"Nach Verarbeitung: {df_limited.shape[0]} Bilder in {df_limited['label'].nunique()} Labels.")

    # Validierung der Verteilung der Labels
    raw_counts = df_raw["label"].value_counts()
    limited_counts = df_limited["label"].value_counts()

    print("\nRohdaten (Top 5 Labels):")
    print(raw_counts.head(5))

    print("\nBegrenzte Daten (Top 5 Labels):")
    print(limited_counts.head(5))

    # Sicherstellen, dass LIMIT_PER_CLASS eingehalten wird
    if LIMIT_PER_CLASS is not None:
        assert (limited_counts <= LIMIT_PER_CLASS).all(), "Fehler: LIMIT_PER_CLASS wurde für mindestens eine Klasse überschritten!"
        print(f"LIMIT_PER_CLASS von {LIMIT_PER_CLASS} Bilder pro Klasse eingehalten.")
    
    # Visualisierung der Label-Verteilungen vor und nach dem Preprocessing
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    sns.barplot(x=raw_counts.head(5).index, y=raw_counts.head(5).values, palette="Blues")
    plt.title("Rohdaten - Top 5 Labels")
    plt.xlabel("Label")
    plt.ylabel("Anzahl von Bildern")
    
    plt.subplot(1, 2, 2)
    sns.barplot(x=limited_counts.head(5).index, y=limited_counts.head(5).values, palette="Greens")
    plt.title("Begrenzte Daten - Top 5 Labels")
    plt.xlabel("Label")
    plt.ylabel("Anzahl von Bildern")
    
    plt.tight_layout()
    plt.show()


### Ausführen: Einlesen + Begrenzung

Jetzt laden wir die Dateipfade, führen das Preprocessing aus und lassen die Validierung laufen.

<hr style="border: 1px solid #ccc; margin: 20px 0;" />


In [ ]:
# Sammle alle PNG-Bildpfade aus dem Basisverzeichnis
image_paths = list(base_path.glob("*.png"))

# Sicherheitscheck
assert len(image_paths) > 0, f"Keine .png Dateien gefunden in {base_path}"

# Erzeuge DataFrames 
df_raw, df_limited = preprocess_images(image_paths)

# Ausgabe der Datenmengen vor und nach der Filterung
print("Rohdaten:", df_raw.shape)
print("Nach Top5/Limit:", df_limited.shape)


### Bilder einheitlich verarbeiten + speichern

Ziel:
- Einheitliche Bildgröße / Farbmodus
- Speichern in `processed_dir`
- Ergebnis ist `processed_df` mit Pfaden zu den verarbeiteten Bildern

<hr style="border: 1px solid #ccc; margin: 20px 0;" />


In [ ]:
def process_and_save_images(df: pd.DataFrame) -> pd.DataFrame:
    
    # Logdatei für fehlerhafte Bilder (wird bei erneutem Lauf zurückgesetzt)
    error_log = processed_dir / "error_log.txt"
    if error_log.exists():
        error_log.unlink()

    out_paths = []
    n_errors = 0

    # Iteriere über alle Bilder und verarbeite sie einheitlich
    for row in tqdm(df.itertuples(index=False), total=len(df), desc="Bilder verarbeiten"):
        raw_path = pathlib.Path(row.path_raw)
        label = row.label

        # Zielordner pro Klasse
        label_dir = processed_dir / label
        label_dir.mkdir(parents=True, exist_ok=True)

        out_path = label_dir / raw_path.name

        try:
            # Öffnen, Farbkonvertierung, Resizing und Speichern
            with Image.open(raw_path) as img:
                img = img.convert(COLOR_MODE)
                img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
                img.save(out_path)

            out_paths.append(str(out_path))

        except Exception as e:
            # Fehler protokollieren, Bild überspringen
            n_errors += 1
            out_paths.append(None)
            with open(error_log, "a", encoding="utf-8") as f:
                f.write(f"{raw_path.resolve()} :: {type(e).__name__}: {e}\n")

    # Erzeuge neuen DataFrame mit Pfaden zu verarbeiteten Bildern
    df2 = df.copy()
    df2["path_processed"] = out_paths
    df2 = df2.dropna(subset=["path_processed"]).reset_index(drop=True)

    # Kurze Zusammenfassung
    print(f"Verarbeitung fertig. Fehler: {n_errors}. Erfolgreich: {len(df2)}.")
    if n_errors > 0:
        print(f"Siehe Log: {error_log}")

    return df2


### Qualitätscheck: verarbeitete Bilder

Wir prüfen stichprobenartig:
- Datei existiert
- Bild kann geöffnet werden
- einfache Visualisierung einiger Samples

<hr style="border: 1px solid #ccc; margin: 20px 0;" />

In [ ]:
def validate_processed_images(df_processed: pd.DataFrame, original_df: pd.DataFrame):
    
    print("\nValidierung der Bildverarbeitung:")

    # 1. Konsistenzprüfung der Anzahl der Bilder
    assert len(df_processed) <= len(original_df), "Fehler: Es wurden mehr Bilder verarbeitet als ursprünglich vorhanden!"
    print(f"Konsistenz der Anzahl: {len(df_processed)} von {len(original_df)} Bildern erfolgreich verarbeitet.")

    # 2. Prüfe, ob alle verarbeiteten Pfade gültig sind
    invalid_paths = [f for f in df_processed["path_processed"] if not pathlib.Path(f).exists()]
    assert not invalid_paths, f"Fehler: {len(invalid_paths)} verarbeitete Bildpfade existieren nicht!"
    print("Alle verarbeiteten Bildpfade existieren und sind erreichbar.")

    # 3. Prüfe auf das gleiche Label
    original_labels = set(original_df["label"].unique())
    processed_labels = set(df_processed["label"].unique())
    assert original_labels == processed_labels, f"Labels sind inkonsistent nach Verarbeitung (Fehlende: {original_labels - processed_labels})!"
    print("Labels nach Verarbeitung konsistent.")

    # 4. Visualisierung von zufälligen Beispielen (dynamischer Seed)
    print("\nStichprobe der verarbeiteten Bilder:")
    dynamic_seed = random.randint(0, 10000)  # Generiere einen dynamischen Seed
    samples = df_processed.sample(n=min(3, len(df_processed)), random_state=dynamic_seed)  # Zufällige Auswahl
    for i, row in enumerate(samples.itertuples(), start=1):
        path = pathlib.Path(row.path_processed)
        try:
            with Image.open(path) as img:
                plt.figure(figsize=(5, 5))
                plt.imshow(img)
                plt.title(f"Verarbeitetes Bild {i} - Label: {row.label}")
                plt.axis("off")
                plt.show()
        except Exception as e:
            print(f"Fehler beim Öffnen von Bild {path}: {e}")


In [ ]:
# Aufruf der Validierungsfunktion:
processed_df = process_and_save_images(df_limited)
validate_processed_images(processed_df, df_limited)


### Train/Val/Test Split

Wir splitten stratifiziert nach Label, damit die Klassenverteilung in allen Splits möglichst ähnlich bleibt.

<hr style="border: 1px solid #ccc; margin: 20px 0;" />


In [ ]:
def create_splits(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    
    # Validierung der Split-Anteile
    assert 0 < TEST_SIZE < 1 and 0 < VAL_SIZE < 1 and (TEST_SIZE + VAL_SIZE) < 1

    # 1) Train vs. temporärer Split (Validation + Test), stratifiziert nach Label
    temp_size = VAL_SIZE + TEST_SIZE
    train_df, temp_df = train_test_split(
        df,
        test_size=temp_size,
        random_state=RANDOM_STATE,
        stratify=df["label"]
    )

    # 2) Aufteilung des temporären Splits in Validation und Test
    test_ratio_in_temp = TEST_SIZE / temp_size
    val_df, test_df = train_test_split(
        temp_df,
        test_size=test_ratio_in_temp,
        random_state=RANDOM_STATE,
        stratify=temp_df["label"]
    )

    # Persistieren der Splits für spätere Modellnotebooks
    train_df.to_csv(splits_dir / "train.csv", index=False)
    val_df.to_csv(splits_dir / "val.csv", index=False)
    test_df.to_csv(splits_dir / "test.csv", index=False)

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )

# Erzeuge Train-/Val-/Test-Splits
train_df, val_df, test_df = create_splits(processed_df)

# Kurzer Überblick über die Split-Größen
print("Split sizes:", len(train_df), len(val_df), len(test_df))


### Split-Checks + Verteilungen

Wir vergleichen die Labelverteilung in:
- begrenzten Daten (`df_limited`)
- Train / Val / Test

Ziel: keine “kaputten” Splits (fehlende Klassen) und ähnliche Verteilungen.

<hr style="border: 1px solid #ccc; margin: 20px 0;" />


In [ ]:
def visualize_label_distribution(*dataframes, names):
    
    # Anzahl der zu vergleichenden DataFrames
    num_dfs = len(dataframes)
    plt.figure(figsize=(5 * num_dfs, 6))
    
    # Erzeuge für jeden DataFrame ein Balkendiagramm der Labelverteilung
    for i, (df, name) in enumerate(zip(dataframes, names), start=1):
        counts = df["label"].value_counts()

        plt.subplot(1, num_dfs, i)
        sns.barplot(x=counts.index, y=counts.values, palette="viridis")
        plt.title(f"Label-Verteilung - {name}")
        plt.xticks(rotation=45)
        plt.xlabel("Labels")
        plt.ylabel("Anzahl")
        
    plt.tight_layout()
    plt.show()

# Vergleich der Labelverteilungen über alle Pipeline-Stufen hinweg
visualize_label_distribution(
    df_raw, df_limited, train_df, val_df, test_df, 
    names=["Rohdaten", "Begrenzte Daten", "Train-Split", "Val-Split", "Test-Split"]
)


In [ ]:
def print_split_stats(name, df):
    print(f"\n{name}: {len(df)}")
    print(df["label"].value_counts())

print_split_stats("Train", train_df)
print_split_stats("Val", val_df)
print_split_stats("Test", test_df)

# Konsistenz
assert len(processed_df) == len(train_df) + len(val_df) + len(test_df)
assert set(train_df["label"].unique()) == set(val_df["label"].unique()) == set(test_df["label"].unique())
print("\n✅ Splits konsistent.")


## Feature-Extraktion (DT / kNN / FFNN)

Für klassische ML-Modelle brauchen wir flache Feature-Vektoren:
- Graustufen + Flatten (für DT/kNN)
- RGB + Flatten (für FFNN)

Außerdem encoden wir die Labels zu Integer-Klassen.

<hr style="border: 1px solid #ccc; margin: 20px 0;" />


In [ ]:
def extract_features(df: pd.DataFrame, img_size: int, mode: str, desc: str):
    
    # Extrahiere Bildpfade und Labels
    paths = df["path_processed"].astype(str).to_numpy()
    y = df["label"].to_numpy()

    # Bestimme Feature-Dimension abhängig vom Farbmodus
    channels = 3 if mode.upper() == "RGB" else 1
    n_features_expected = img_size * img_size * channels

    # Vorallokation des Feature-Arrays
    X = np.empty((len(df), n_features_expected), dtype=np.float32)

    # Lade, skaliere und flachere jedes Bild
    for i, p in enumerate(tqdm(paths, desc=desc)):
        with Image.open(p) as img:
            img = img.convert(mode).resize((img_size, img_size), Image.BILINEAR)
            arr = np.asarray(img, dtype=np.float32) / 255.0
            X[i] = arr.reshape(-1)

    return X, y


# Sanity Check: Feature-Extraktion auf kleinem Beispiel
X_tmp, y_tmp = extract_features(
    test_df,
    img_size=IMG_SIZE_FLAT,
    mode="L",
    desc="Feature-Test"
)

print("X_tmp shape:", X_tmp.shape, "y_tmp:", y_tmp.shape)
print("Beispiel Feature-Werte:", X_tmp[0][:20])

# Konsistenz-Checks (Numerik und Wertebereich)
assert np.isfinite(X_tmp).all()
assert (X_tmp.min() >= 0) and (X_tmp.max() <= 1)
print("Feature-Test ok.")


### Label-Encoding

Die Modelle arbeiten mit numerischen Klassenlabels.
Wir nutzen `LabelEncoder`, damit die Zuordnung reproduzierbar bleibt.

<hr style="border: 1px solid #ccc; margin: 20px 0;" />


In [ ]:
le = LabelEncoder()
y_train_enc = le.fit_transform(train_df["label"])
y_val_enc = le.transform(val_df["label"])
y_test_enc = le.transform(test_df["label"])

print("Klassen:", list(le.classes_))


### Feature-Qualitätscheck

Wir prüfen:
- Shapes passen
- Labels passen
- Wertebereich ist [0,1]

<hr style="border: 1px solid #ccc; margin: 20px 0;" />


In [ ]:
def validate_features(X, y, img_size, mode, label_encoder):
    
    print("\nValidierung der Feature-Extraktion:")

    # 1. Prüfe auf richtige Dimensionalität
    n_features_expected = img_size * img_size * (3 if mode.upper() == "RGB" else 1)
    assert X.shape[1] == n_features_expected, f"Fehler: Erwartet {n_features_expected} Features, aber erhalten: {X.shape[1]}"
    print("Dimensionalität der Features korrekt:", X.shape)

    # 2. Prüfe auf NaNs und Wertebereich
    assert np.isfinite(X).all(), "Fehler: Features enthalten NaNs oder unendliche Werte!"
    assert X.min() >= 0 and X.max() <= 1, f"Fehler: Features außerhalb des Bereichs [0, 1] gefunden! (min: {X.min()}, max: {X.max()})"
    print("Features enthalten keine NaNs und die Werte liegen im Bereich [0, 1].")

    # 3. Prüfe die Label-Kodierungen
    assert len(y) == X.shape[0], "Fehler: Anzahl der Labels stimmt nicht mit den Anzahl der Samples überein!"
    assert len(label_encoder.classes_) > 0, "Fehler: LabelEncoder hat keine Klassen!"
    print("Labels wurden korrekt kodiert:", list(label_encoder.classes_))

    # 4. Statistische Analyse der extrahierten Features
    mean_per_sample = X.mean(axis=1)
    std_per_sample = X.std(axis=1)
    print(f"Durchschnittlicher Feature-Mittelwert pro Bild: {mean_per_sample.mean():.3f}")
    print(f"Durchschnittliche Feature-Standardabweichung pro Bild: {std_per_sample.mean():.3f}")

    # Histogram für Feature-Werte
    plt.figure(figsize=(10, 6))
    sns.histplot(mean_per_sample, kde=True, bins=30, color="blue", label="Mittelwerte")
    sns.histplot(std_per_sample, kde=True, bins=30, color="orange", label="Standardabweichungen")
    plt.legend()
    plt.title("Verteilung der Feature-Mittelwerte und Standardabweichungen")
    plt.show()

# Aufruf der `extract_features`-Funktion und anschließende Validierung
X_tmp, y_tmp = extract_features(test_df, img_size=IMG_SIZE_FLAT, mode="L", desc="Feature-Test")
validate_features(X_tmp, y_tmp, img_size=IMG_SIZE_FLAT, mode="L", label_encoder=le)

### Features exportieren

Wir speichern Features getrennt für:
- Decision Tree (Graustufen, klein)
- kNN (skaliert + PCA)
- FFNN (RGB, mittel)

Damit können die Modellnotebooks direkt darauf zugreifen.

<hr style="border: 1px solid #ccc; margin: 20px 0;" />


In [ ]:
# Feature-Extraktion für Decision Tree (Graustufen, flach)
X_train_dt, y_train_dt = extract_features(train_df, IMG_SIZE_FLAT, mode="L", desc="DT Train")
X_val_dt, y_val_dt = extract_features(val_df, IMG_SIZE_FLAT, mode="L", desc="DT Val")
X_test_dt, y_test_dt = extract_features(test_df, IMG_SIZE_FLAT, mode="L", desc="DT Test")

# Speichern der DT-Features und Labels für spätere Modell-Notebooks
np.savez_compressed(
    features_dir / "dt_features.npz",
    X_train=X_train_dt, X_val=X_val_dt, X_test=X_test_dt,
    y_train=y_train_enc, y_val=y_val_enc, y_test=y_test_enc
)

print("dt_features.npz gespeichert.")


In [ ]:
# Standardisierung der DT-Features (wichtig für Distanz-basierte Modelle)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_dt)
X_val_scaled = scaler.transform(X_val_dt)
X_test_scaled = scaler.transform(X_test_dt)

# PCA zur Dimensionsreduktion (95 % erklärte Varianz)
pca = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_train_knn = pca.fit_transform(X_train_scaled)
X_val_knn = pca.transform(X_val_scaled)
X_test_knn = pca.transform(X_test_scaled)

# Speichern der kNN-Features inkl. PCA-Informationen
np.savez_compressed(
    features_dir / "knn_features.npz",
    X_train=X_train_knn, X_val=X_val_knn, X_test=X_test_knn,
    y_train=y_train_enc, y_val=y_val_enc, y_test=y_test_enc,
    explained_variance_ratio=pca.explained_variance_ratio_
)

print(f"PCA Varianz (Summe): {pca.explained_variance_ratio_.sum()*100:.2f}%")
print("knn_features.npz gespeichert.")


In [ ]:
# Feature-Extraktion für FFNN (RGB, mittlere Auflösung, flach)
X_train_ffnn, _ = extract_features(train_df, IMG_SIZE_FFNN, mode="RGB", desc="FFNN Train")
X_val_ffnn, _ = extract_features(val_df, IMG_SIZE_FFNN, mode="RGB", desc="FFNN Val")
X_test_ffnn, _ = extract_features(test_df, IMG_SIZE_FFNN, mode="RGB", desc="FFNN Test")

# Speichern der FFNN-Features und Labels
np.savez_compressed(
    features_dir / "ffnn_features.npz",
    X_train=X_train_ffnn, X_val=X_val_ffnn, X_test=X_test_ffnn,
    y_train=y_train_enc, y_val=y_val_enc, y_test=y_test_enc
)

print("ffnn_features.npz gespeichert.")


## CNN Input Pipeline (`tf.data`)

Für das CNN verwenden wir keine flachen Features, sondern laden Bilder on-the-fly.
Das ist speichereffizient und erlaubt Data Pipeline Optimierung (Cache/Prefetch).

<hr style="border: 1px solid #ccc; margin: 20px 0;" />



In [ ]:
# CNN DATASETS (tf.data)

AUTOTUNE = tf.data.AUTOTUNE

# Datenaugmentation für das Training (Werte werden anschließend geclippt)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomZoom(0.1),
], name="data_augmentation")


def decode_and_resize(path, label):

    #Lädt PNG, decodiert es, skaliert auf [0,1] und resized auf IMG_SIZE.

    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_png(img_bytes, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)  # garantiert [0,1]
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return img, label


def augment_and_clip(img, label):
    # Wende Augmentation an und clippe Werte zurück in [0,1]
    img = data_augmentation(img, training=True)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label


def make_dataset(df: pd.DataFrame, labels: np.ndarray, training: bool = False) -> tf.data.Dataset:
    # Erstellt ein tf.data Dataset aus DataFrame und Labels.
    # Optional mit Shuffle und Augmentation für das Training.

    paths = df["path_processed"].astype(str).to_numpy()
    labels = labels.astype(np.int64)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(decode_and_resize, num_parallel_calls=AUTOTUNE)

    if training:
        ds = ds.shuffle(
            buffer_size=len(df),
            seed=RANDOM_STATE,
            reshuffle_each_iteration=True
        )
        ds = ds.map(augment_and_clip, num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


# Erzeuge Train-/Val-/Test-Datasets für das CNN
train_ds = make_dataset(train_df, y_train_enc, training=True)
val_ds   = make_dataset(val_df,   y_val_enc,   training=False)
test_ds  = make_dataset(test_df,  y_test_enc,  training=False)

print("TF Datasets erstellt: train_ds / val_ds / test_ds")


### Qualitätsprüfung (Stichprobe)

Wir visualisieren eine Batch aus dem Training:
- stimmt das Format?
- stimmen Labels?
- sehen die Bilder “normal” aus?

<hr style="border: 1px solid #ccc; margin: 20px 0;" />

In [ ]:
# Qualitätsprüfung 

imgs, labs = next(iter(train_ds))

print("Batch images:", imgs.shape)   # (BATCH_SIZE, 224, 224, 3)
print("Batch labels:", labs.shape)   # (BATCH_SIZE,)

assert imgs.shape[1:3] == (IMG_SIZE, IMG_SIZE), "Fehler: Auflösung falsch!"
assert imgs.shape[-1] == 3, "Fehler: Farbkanäle falsch!"
assert float(tf.reduce_min(imgs).numpy()) >= 0.0, "Fehler: min(imgs) < 0!"
assert float(tf.reduce_max(imgs).numpy()) <= 1.0, "Fehler: max(imgs) > 1!"

print("CNN Dataset ok (Werte in [0,1])")

# Beispielbild anzeigen
plt.figure(figsize=(5, 5))
plt.imshow(imgs[0].numpy())
plt.title(f"Train example - class idx: {int(labs[0].numpy())}")
plt.axis("off")
plt.show()

## 4. Konsistenzprüfung der gesamten Pipeline

Zum Schluss prüfen wir, ob die Datenmengen zwischen den Schritten zusammenpassen:
Rohdaten → begrenzte Daten → verarbeitete Daten → Splits.

<hr style="border: 1px solid #ccc; margin: 20px 0;" />

In [ ]:
def check_pipeline_consistency(raw_count, limited_count, processed_count, train_count, val_count, test_count):

    print("\nAbschlussprüfung der Pipeline:")
    assert raw_count >= limited_count, "Fehler: Begrenzung erzeugte mehr Bilder als vorhanden!"
    assert limited_count == processed_count, "Fehler: Begrenzte und verarbeitete Bilder stimmen nicht überein!"
    assert processed_count == (train_count + val_count + test_count), \
        "Fehler: Summe der Splits stimmt nicht mit den verarbeiteten Daten überein!"

    print("Pipeline konsistent:")
    print(f"Rohdaten: {raw_count}, Begrenzte Daten: {limited_count}")
    print(f"Verarbeitete Daten: {processed_count}, Train: {train_count}, Val: {val_count}, Test: {test_count}")

# Aufruf:
check_pipeline_consistency(
    raw_count=len(df_raw), limited_count=len(df_limited), processed_count=len(processed_df),
    train_count=len(train_df), val_count=len(val_df), test_count=len(test_df)
)